# Example steps and codes running hypoddpy for earthquake relocation

In [1]:
#import needed packages.
""" 
    Download hypoDD at https://www.ldeo.columbia.edu/~felixw/hypoDD.html
    Or use the copy included in this python interface package.
"""
import os, glob,time
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import HypoDDCore as HDC


In [2]:
def tic():
    return time.perf_counter()

def toc(t0, label):
    dt = time.perf_counter() - t0
    print(f"[TIMER] {label}: {dt:.2f} s")

In [ ]:
t_total = tic()
print("[TIMER] HypoDD started")

binpath='/Users/xtyang/bin'
indir='input'
outdir='output'
if not os.path.exists(outdir):
    os.makedirs(outdir)
if not os.path.exists(indir):
    os.makedirs(indir)
namebase='eq_ct'
station_file='input/station_eg.csv'
station_file_reformat='input/station.dat'
phase_file='input/eg_hyp_full.pha' #input original phase file.
phase_file_reformat=f'{indir}/{namebase}.pha' #output reformatted phase file for hypoDD.

dep_corr = 5

#subset params
time_range='20190704-20190710'
lat_range = [35.45,36.05]
lon_range = [-117.8,-117.25]

# -------------------------------
# Config
# -------------------------------
hdconfig = HDC.HypoDDConfig(
    binpath=binpath,
    indir=indir,
    outdir=outdir,
    namebase=namebase,
    station_file=station_file,
    phase_file=phase_file_reformat
)

# -------------------------------
# 1. Format station & phase
# -------------------------------
pha_dict = HDC.load_phasedata(phase_file)

HDC.reformat_stationfile(station_file, station_file_reformat)

t0 = tic()
_,evid_list =HDC.reformat_phasefile(hdconfig, phase_file, time_range=time_range, lat_range=lat_range, lon_range=lon_range)
toc(t0, "Reformat phase file")

# -------------------------------
# 2. ph2dt
# -------------------------------
t0 = tic()
HDC.run_ph2dt(hdconfig)
toc(t0, "ph2dt")

# -------------------------------
# 3. hypoDD (core relocation)
# -------------------------------

hdcore = HDC.HypoDDCore(hdconfig, evid_list, pha_dict)

t0 = tic()
print(f"Running hypoDD ")
hdcore.run()
toc(t0, f"Run hypoDD")

# -------------------------------
# 4. Cleanup
# -------------------------------
reloc_grids = glob.glob(f'output/hypoDD_{hdconfig.namebase}.reloc.*')

for f in reloc_grids:
    if os.path.exists(f):
        os.unlink(f)
# -------------------------------
# Total time
# -------------------------------
toc(t_total, "TOTAL HypoDD run")

[TIMER] HypoDD started
[INFO] Wrote single phase file: input/eq_ct.pha
[INFO] Number of events: 10755
[TIMER] Reformat phase file: 2.40 s
run ph2dt (single run)
[INFO] ph2dt completed: output/eq_ct.ph2dt
[TIMER] ph2dt: 16.34 s
Running hypoDD 
[TIMER] Run hypoDD: 34.19 s
